In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_invoices
# Source          : payments.csv
# Target          : procurement.bronze.bronze_invoices
# Audit Table     : procurement.audit.duplicate_invoices
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw invoices master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load  Invoices master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Payment IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_INVOICES)
print(AUDIT_DUPLICATE_INVOICES)
print(INVOICES_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Invoices schema
invoices_schema = StructType([
    StructField("invoice_id", StringType(), False),
    StructField("invoice_date", StringType(), True),
    StructField("po_id", StringType(), True),
    StructField("po_amount_expected", DecimalType(18,2), True),
    StructField("invoice_amount", DecimalType(18,2), True),
    StructField("currency", StringType(), True),
    StructField("due_date", StringType(), True),
    StructField("invoice_status", StringType(), True)  
])
# Read Invoices master data from landing volume
bronze_invoices_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(invoices_schema)
    .load(INVOICES_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_invoices_df.count()}")

print("\nSchema:")
bronze_invoices_df.printSchema()

print("\nColumns:")
print(bronze_invoices_df.columns)

print("\nSampledata:")
display(bronze_invoices_df.limit(10))

In [0]:
# Check the NULL and Blank invoice_ids
null_blank_invoice_id= bronze_invoices_df.filter(col("invoice_id").isNull() | (trim(col("invoice_id")) == ""))

print(f"Total NULL or Blank invoice_ids : {null_blank_invoice_id.count()}")

display(null_blank_invoice_id)

In [0]:
#Check the NULL and Blank invoice date
null_blank_invoice_date = bronze_invoices_df.filter(col("invoice_date").isNull() | (trim(col("invoice_date")) == ""))

print(f"Total NULL or Blank invoice_date : {null_blank_invoice_date.count()}")

display(null_blank_invoice_date)

In [0]:
# Check the NULL and Blank po_ids
null_blank_po_id = bronze_invoices_df.filter(col("po_id").isNull() | (trim(col("po_id")) == ""))

print(f"Total NULL or Blank po_ids : {null_blank_po_id.count()}")

display(null_blank_po_id)

In [0]:
#Check the negative or NULL po_amount_expected
negative_po_amount_expected = bronze_invoices_df.filter((col("po_amount_expected") < 0) | (col("po_amount_expected").isNull()))

print(f"Total Null and negative amounts : {negative_po_amount_expected.count()}")

display(negative_po_amount_expected)

In [0]:
#Check the negative or NULL invoice_amount
negative_invoice_amount = bronze_invoices_df.filter((col("invoice_amount") < 0) | (col("invoice_amount").isNull()))

print(f"Total Null and negative invoice_amount : {negative_invoice_amount.count()}")

display(negative_invoice_amount)


In [0]:
#Check the NULL and Blank currency
null_blank_currency = bronze_invoices_df.filter(col("currency").isNull() | (trim(col("currency")) == ""))

print(f"Total NULL or Blank currency : {null_blank_currency.count()}")

display(null_blank_currency)

In [0]:
#Check the NULL and Blank due_date
null_blank_due_date = bronze_invoices_df.filter(col("due_date").isNull() | (trim(col("due_date")) == ""))

print(f"Total NULL or Blank due_date : {null_blank_due_date.count()}")

display(null_blank_due_date)

In [0]:
#Check the NULL and Blank invoice_status
null_blank_invoice_status= bronze_invoices_df.filter(col("invoice_status").isNull() | (trim(col("invoice_status")) == ""))

print(f"Total NULL or Blank payment_status : {null_blank_invoice_status.count()}")

display(null_blank_invoice_status)

In [0]:
# ============================================================
# Identify Duplicate Invoice IDs igonere NULLs
# ============================================================

duplicate_invoice_keys = (
    bronze_invoices_df
    .filter(
        col("invoice_id").isNotNull() &
        (trim(col("invoice_id")) != "")
    )
    .groupBy("invoice_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_invoice_keys)

In [0]:
# ============================================================
# Identify Duplicate Invoices Records
# Business Rule: Keep the first occurrence of each Invoices ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("invoice_id").orderBy("invoice_date")

invoice_rank_df = (
    bronze_invoices_df
        .join(
            duplicate_invoice_keys.select("invoice_id"),
            on="invoice_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)

display(invoice_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate Invoices Records
# ============================================================

duplicate_invoices = (
    invoice_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Invoices Records : {duplicate_invoices.count()}")
display(duplicate_invoices)


In [0]:
# ============================================================
# Add Audit Metadata for duplicate invoice IDs 
# ============================================================

duplicate_invoices = (
    duplicate_invoices
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Invoices"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_invoices)

In [0]:
# ============================================================
# Add Audit Metadata for null invoice IDs 
# ============================================================

invalid_invoices = (
    null_blank_invoice_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Invoices"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(invalid_invoices)

In [0]:
    # ============================================================
    # Write Duplicate Records to Audit Table
    # ============================================================

    duplicate_count = duplicate_invoices.count()

    if duplicate_count > 0:

        write_delta(
            df = duplicate_invoices,
            table_name = AUDIT_DUPLICATE_INVOICES
        )

        print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_INVOICES}")

    else:

        print("No duplicate Invoices records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid Invoice IDs to Audit Table
# ============================================================

invalid_count = invalid_invoices.count()

if invalid_count > 0:

    write_delta(
        df = invalid_invoices,
        table_name = AUDIT_INVALID_INVOICES
    )

    print(f"Successfully written {invalid_count} invalid invoice record(s) to {AUDIT_INVALID_INVOICES}")

else:

    print("No invalid invoice records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_invoices_final_df = (
    bronze_invoices_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Invoices.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_invoices_final_df,
    table_name=BRONZE_INVOICES
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_invoices = spark.table(BRONZE_INVOICES)

print(f"Total Bronze Records : {bronze_invoices.count()}")

display(bronze_invoices)

In [0]:
# ============================================================
# Bronze invoices complete summary
# ============================================================
print("=" * 60)
print("Bronze Invoice Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_invoices_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_invoices.count()}")

print(f"{'Invalid Invoice Records':<30}: {invalid_invoices.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_invoices.count() + invalid_invoices.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_INVOICES).count()}")